# Persistent Memory

*Notebook 20*

In-memory sessions disappear when the kernel restarts. Persistent memory survives — storing facts, preferences, and history in SQLite so agents remember users across sessions.
<br>
<br>

**Topics:**

- Short-term vs long-term memory — when each applies

- Persistent SQLite sessions — surviving restarts

- Storing user facts and preferences across sessions

- Avoiding noisy memory — when to summarize vs drop

- Stale memory cleanup and correcting wrong memories

- Conflicting memories and user-controlled forgetting

- What NOT to store — privacy and sensitive data boundaries

---

## 🔧 Setup

In [ ]:
from pathlib import Path
from dotenv import load_dotenv
load_dotenv(dotenv_path=Path("..") / ".env")

from agents import Agent, Runner, SQLiteSession

MODEL = "gpt-5-mini"

print("✅ Ready!")

---

## 🎯 The Problem

In-memory sessions disappear when the kernel restarts — any context the agent built up is gone.

Persistent memory solves this by writing session state to disk so agents can retain useful facts and preferences across sessions.

---

## 🧠 Short-Term vs Long-Term Memory

In Lesson 19, sessions stored conversation history in memory — fast and automatic, but gone when the kernel restarts.

**Short-term (in-memory):** No setup, no persistence. Lost on restart. Best for single conversations and demos.

**Long-term (persistent):** Survives restarts. Stored on disk. Best for user preferences, facts, and history that should follow a user across days or sessions.

The difference is one parameter: `db_path`.

---

## 💾 Part 1: Persistent Sessions with SQLite

Pass a `db_path` to `SQLiteSession` and memory is written to disk automatically after every turn.

In [ ]:
# Create a persistent session — written to disk after every turn
session = SQLiteSession(
    "user_alex",
    db_path="memory.sqlite"
)

assistant = Agent(
    name="Assistant",
    instructions="You are a helpful assistant. Remember useful facts users share about themselves.",
    model=MODEL
)

# Turn 1: tell the agent something to remember

result = await Runner.run(assistant, input="My name is Alex and I'm learning Python.", session=session)

print("Turn 1:", result.final_output)

#### After a Restart — Same Session ID, Same Memory

In [ ]:
# Simulate a restart — recreate the session with the same ID and path
# In a real restart you would just run this cell and the agent would remember
reloaded_session = SQLiteSession(
    "user_alex",            # same ID
    db_path="memory.sqlite"  # same file
)

result = await Runner.run(assistant, input="What do you know about me?", session=reloaded_session)

print("After reload:", result.final_output)

### 💡 Why This Works

The session ID (`"user_alex"`) is the key.

Recreating a `SQLiteSession` with the same ID and `db_path` loads the full conversation history from disk — the agent picks up exactly where it left off.

---

## 📋 Part 2: Storing User Preferences

Persistent memory is especially useful for preferences — things the user tells the agent once that should apply to every future interaction.

In [ ]:
prefs_session = SQLiteSession(
    "user_prefs_demo",
    db_path="preferences.sqlite"
)

prefs_instructions = (
    "You are a helpful writing assistant.\n"
    "Remember the user's communication preferences and apply them in every response.\n"
    "If the user hasn't stated preferences yet, ask them before answering."
)

prefs_agent = Agent(
    name="PreferencesAssistant",
    instructions=prefs_instructions,
    model=MODEL
)

# Turn 1: user sets their preferences

result = await Runner.run(prefs_agent, input="I prefer short answers, bullet points over prose, and plain language — no jargon.", session=prefs_session)

print("Turn 1:", result.final_output)

#### Apply Preferences in a Later Session

In [ ]:
# Turn 2: later session — preferences are remembered
later_session = SQLiteSession(
    "user_prefs_demo",
    db_path="preferences.sqlite"
)

result = await Runner.run(prefs_agent, input="Explain what a REST API is.", session=later_session)

print("Turn 2 (preferences applied):", result.final_output)

### 💡 Why This Works

The agent's instructions tell it to remember and apply preferences.

The persistent session stores the full conversation including the preference-setting turn — so the agent always has that context available, regardless of when it was set.

---

## ⚠️ Part 3: Avoiding Noisy Memory

Treat memory as a curated record rather than an exhaustive log — storing everything indiscriminately leads to contradictions, stale facts, and context bloat.

Even a clean session like the one below grows unbounded — every turn costs tokens, and once the history exceeds the model's context window, older facts start dropping silently.

Compress proactively, not reactively.

#### Build a Noisy Session

In [ ]:
# Build a noisy session with many turns
noisy_session = SQLiteSession(
    "user_noisy_demo",
    db_path="noisy.sqlite"
)

messages = [
    "I'm building a Python web scraper.",
    "I want to scrape product prices from e-commerce sites.",
    "I prefer using requests and BeautifulSoup.",
    "I've already handled pagination.",
    "The main challenge left is rate limiting.",
]

for message in messages:

    await Runner.run(assistant, input=message, session=noisy_session)

items_before = await noisy_session.get_items()
print(f"Before cleanup: {len(items_before)} items")

#### Summarize the History

In [ ]:
# Step 1: Summarize — extract only what's worth keeping
summarizer = Agent(
    name="Summarizer",
    instructions="Extract only the key facts worth remembering. Return a concise bullet list.",
    model=MODEL
)

history_text = "\n".join(str(item)[:200] for item in items_before)

summary_result = await Runner.run(summarizer, input=f"Summarize the key facts from this conversation:\n{history_text}")

summary = summary_result.final_output
print(f"Summary:\n{summary}")

#### Clear and Store the Summary

In [ ]:
# Step 2: Clear the session and store only the summary

await noisy_session.clear_session()

# Re-use the same session object — clear_session() doesn't invalidate it
await Runner.run(assistant, input=f"Here is a summary of what we have discussed: {summary}", session=noisy_session)

items_after = await noisy_session.get_items()
print(f"After cleanup: {len(items_after)} items (was {len(items_before)})")

### 💡 Why This Works

The summarize → clear → store pattern compresses a long conversation into a compact set of facts — the agent retains everything that matters in a fraction of the tokens.

Apply this pattern proactively — set a turn count threshold (e.g. every 20 turns) rather than waiting for quality to degrade.

## 🔐 Part 4: What NOT to Store — Privacy and Correction

Persistent memory is powerful but requires discipline.

Not everything belongs in long-term storage, and stored facts can become wrong over time.

#### What to Keep vs Drop

<div style="text-align: left; display: inline-block;">

| Keep | Drop |
|------|------|
| Name, role, goals | Greetings and filler turns |
| Stated preferences | Repeated confirmations |
| Decisions and constraints | Stale intermediate outputs |
| Durable facts | Low-value tool noise |

</div>

⚠️ **Security note:** Never store passwords, API keys, financial account details, or health information in agent memory.

Only persist data the user would expect an assistant to remember.

Prompt injection and input validation are covered in Lesson 23.

In practice, filter sensitive input before passing it to the agent, or use output guardrails (Lesson 22) to detect and block sensitive data from being persisted.

In [ ]:
correction_session = SQLiteSession(
    "user_correction_demo",
    db_path="correction.sqlite"
)

correction_instructions = (
    "You are a helpful assistant. Remember what users tell you about themselves. "
    "If a user corrects an earlier fact, treat the most recent statement as the current truth."
)

correction_agent = Agent(
    name="CorrectionAssistant",
    instructions=correction_instructions,
    model=MODEL
)

# Store an initial fact

result = await Runner.run(correction_agent, input="I work as a data analyst.", session=correction_session)

print(f"Stored: {result.final_output}\n")

# User corrects the fact

result = await Runner.run(correction_agent, input="Actually I just changed jobs — I'm now a machine learning engineer.", session=correction_session)

print(f"Corrected: {result.final_output}\n")

# Verify the agent uses the updated fact

result = await Runner.run(correction_agent, input="What's my current role?", session=correction_session)

print(f"Recalled: {result.final_output}")

#### Conflicting Memory and Forgetting

For repeated conflicts, apply the summarize → clear → store pattern from Part 3 rather than letting contradictions stack.

The SDK doesn't support per-fact deletion — for true forgetting, the practical pattern is to summarize the session, drop the unwanted fact during summarization, then clear and restore.

### 💡 Why This Works

SQLite gives you durability — curation gives you useful memory.

The right mental model is a curated record: keep stable facts and preferences, drop noise and sensitive data, and correct wrong facts promptly rather than letting them accumulate.

#### 🧹 Cleanup

In [ ]:
# Delete the SQLite files created by this notebook.
# These files ARE the persistent storage — delete them for a completely fresh start.
for f in ["memory.sqlite", "preferences.sqlite", "noisy.sqlite", "correction.sqlite"]:
    if Path(f).exists():
        Path(f).unlink()
        print(f"✅ Removed {f}")
print("✅ Cleanup complete")

---

## 💪 Practice Exercises

### Exercise 1: Persistent User Profile

*Covers: persistent memory — storing and retrieving user state*

Build an agent that remembers a user's name, preferences, and goals across sessions.

In [ ]:
# --------------------------------------------------------------
# 💪 Exercise 1: Persistent User Profile
# --------------------------------------------------------------
# Objective: Build an agent that persists user facts across sessions.
#
# TODO 1: Create a SQLiteSession with a db_path
# TODO 2: Create an agent with instructions to remember user facts
# TODO 3: Run 3 turns — introduce yourself, state a preference, ask a question
# TODO 4: Simulate a restart (recreate session with same ID + db_path)
# TODO 5: Confirm the agent still remembers your name and preference

# --- Write your code below this line ---

### Exercise 2: Memory Cleanup

*Covers: memory management — summarize, clear, and store*

Build a session with 5+ turns, then apply the summarize → clear → store pattern.

In [ ]:
# --------------------------------------------------------------
# 💪 Exercise 2: Memory Cleanup
# --------------------------------------------------------------
# Objective: Apply the summarize → clear → store pattern.
#
# TODO 1: Create a session and run 5 turns of conversation on any topic
# TODO 2: Call get_items() and print how many items are stored
# TODO 3: Run a summarizer agent to extract key facts
# TODO 4: Clear the session with clear_session()
# TODO 5: Store only the summary, then confirm the item count dropped

# --- Write your code below this line ---

---

## 🎯 Key Takeaways

**Persistent Memory:**

- `SQLiteSession("id", db_path="file.sqlite")` — memory survives restarts

- Same ID + same file path = same memory across any session

- Change either value and you get a blank slate
<br>
<br>

**What to Store:**

- Facts, preferences, decisions — not raw conversation history

- Think of long-term memory as a curated record, not a log
<br>
<br>

**Avoiding Noise:**

- Session size grows with every turn — quality degrades silently

- Apply summarize → clear → store proactively, not reactively

- `get_items()` lets you inspect and `clear_session()` lets you prune
<br>
<br>

**Privacy and Correction:**

- Never store passwords, API keys, financial details, or health data

- Correct wrong facts promptly — the agent should treat the latest statement as truth

- Use summarize → clear → store to resolve accumulated contradictions
<br>
<br>

**Use SQLite for exact facts — use vector memory for meaning:**

- SQLite is best for storing specific facts, preferences, and decisions — retrieval by exact session ID

- When you need to retrieve memories by meaning rather than exact match, that's covered in Lesson 21.

---

## 📍 Next Step

**Notebook 21: Vector Memory with ChromaDB**  

Store and retrieve memories by meaning rather than exact keywords, using embeddings and semantic search.

---

##### 🔧 [Troubleshooting Guide](https://github.com/barrettscott/openai-agents/blob/main/TROUBLESHOOTING.md#lesson-20-persistent-memory)

---